In [1]:
import torch
from tqdm import tqdm
import pandas as pd
from alphagenome_pytorch import AlphaGenome
from alphagenome_pytorch.variant_scoring import (
    VariantScoringModel, Variant, Interval,
    CenterMaskScorer, OutputType, AggregationType,
    get_recommended_scorers,
)
from alphagenome_pytorch.variant_scoring.scorers.gene_mask import GeneMaskMode


In [ ]:

# Load model (track_means are bundled in weights from convert_weights.py)
model = AlphaGenome.from_pretrained('data/model_related/model_all_folds.safetensors', device='cuda')


In [ ]:
# Create scoring wrapper
scoring_model = VariantScoringModel(
    model,
    fasta_path='data/model_related/hg38.fa',
    gtf_path='data/annotations/gencode.v46.annotation.parquet',
    polya_path='data/annotations/gencode.v46.polyAs.linked.parquet',  # Optional, for PolyadenylationScorer
    default_organism='human',
)
scoring_model.load_all_metadata('track_metadata.parquet')

In [4]:
# load scorere that alphagenome used for eQTL scoring in the paper
from alphagenome_pytorch.variant_scoring import GeneMaskLFCScorer

rna_scorer = GeneMaskLFCScorer(
    requested_output=OutputType.RNA_SEQ,
    mask_mode=GeneMaskMode.EXONS,  # EXONS or BODY
    resolution=1,  # Default 128bp for gene-level
)

In [5]:
def score_variant_pytorch(variant_id):
    variant = Variant.from_str(variant_id, format="gtex")

    interval = Interval.centered_on(
        variant.chromosome,
        variant.position
    )

    scores = scoring_model.score_variant(
        interval=interval,
        variant=variant,
        scorers=[rna_scorer],
        to_cpu=True,
    )

    return scoring_model.tidy_scores(scores)

In [6]:
# @title Eval configs.
url = "https://storage.googleapis.com/alphagenome/evals/clinvar_noncoding_predictions.feather"
df = pd.read_feather(url)

df.head()


,variant_id,prediction,target,variant_scorer,output_type,metric_calculator,metric_name
0,chr1_1043197_G_C_hg38,0.016315,1.0,"ensemble_scorer;sj,ssu,ss",RNA_SEQ;SPLICE_SITE_POSITIONS;SPLICE_SITE_USAG...,SimpleAggregatorAUPRCCalculator,auprc_max_abs_track_aggregation
1,chr1_2520467_C_T_hg38,0.006831,1.0,"ensemble_scorer;sj,ssu,ss",RNA_SEQ;SPLICE_SITE_POSITIONS;SPLICE_SITE_USAG...,SimpleAggregatorAUPRCCalculator,auprc_max_abs_track_aggregation
2,chr1_9943525_C_T_hg38,0.040409,1.0,"ensemble_scorer;sj,ssu,ss",RNA_SEQ;SPLICE_SITE_POSITIONS;SPLICE_SITE_USAG...,SimpleAggregatorAUPRCCalculator,auprc_max_abs_track_aggregation
3,chr1_11790916_C_T_hg38,0.811025,1.0,"ensemble_scorer;sj,ssu,ss",RNA_SEQ;SPLICE_SITE_POSITIONS;SPLICE_SITE_USAG...,SimpleAggregatorAUPRCCalculator,auprc_max_abs_track_aggregation
4,chr1_11998739_T_G_hg38,0.343018,1.0,"ensemble_scorer;sj,ssu,ss",RNA_SEQ;SPLICE_SITE_POSITIONS;SPLICE_SITE_USAG...,SimpleAggregatorAUPRCCalculator,auprc_max_abs_track_aggregation


In [7]:
def parse_variant(variant_id):
    parts = variant_id.split("_")
    if len(parts) >= 4:
        chrom, pos, ref, alt = parts[:4]
        return ref, alt
    return None, None

def classify_variant(variant_id):
    ref, alt = parse_variant(variant_id)
    
    if ref is None:
        return "unknown"
    
    if len(ref) == len(alt) == 1:
        return "snp"
    elif len(alt) > len(ref):
        return "insertion"
    elif len(ref) > len(alt):
        return "deletion"
    else:
        return "other"
    
df["variant_type"] = df["variant_id"].apply(classify_variant)

insertions = df[df["variant_type"] == "insertion"].drop_duplicates("variant_id").head(100)
deletions = df[df["variant_type"] == "deletion"].drop_duplicates("variant_id").head(100)

variant_insertions = insertions["variant_id"].tolist()
variant_deletions = deletions["variant_id"].tolist()

In [8]:
insertions.head()

,variant_id,prediction,target,variant_scorer,output_type,metric_calculator,metric_name,variant_type
141,chr2_60461136_G_GCCA_hg38,0.070085,1.0,"ensemble_scorer;sj,ssu,ss",RNA_SEQ;SPLICE_SITE_POSITIONS;SPLICE_SITE_USAG...,SimpleAggregatorAUPRCCalculator,auprc_max_abs_track_aggregation,insertion
151,chr2_85662080_G_GGGC_hg38,0.123210,1.0,"ensemble_scorer;sj,ssu,ss",RNA_SEQ;SPLICE_SITE_POSITIONS;SPLICE_SITE_USAG...,SimpleAggregatorAUPRCCalculator,auprc_max_abs_track_aggregation,insertion
152,chr2_86281698_A_AT_hg38,0.057882,1.0,"ensemble_scorer;sj,ssu,ss",RNA_SEQ;SPLICE_SITE_POSITIONS;SPLICE_SITE_USAG...,SimpleAggregatorAUPRCCalculator,auprc_max_abs_track_aggregation,insertion
208,chr2_202542301_A_AGGG_hg38,1.187738,1.0,"ensemble_scorer;sj,ssu,ss",RNA_SEQ;SPLICE_SITE_POSITIONS;SPLICE_SITE_USAG...,SimpleAggregatorAUPRCCalculator,auprc_max_abs_track_aggregation,insertion
217,chr2_219421371_T_TGGA_hg38,0.148407,1.0,"ensemble_scorer;sj,ssu,ss",RNA_SEQ;SPLICE_SITE_POSITIONS;SPLICE_SITE_USAG...,SimpleAggregatorAUPRCCalculator,auprc_max_abs_track_aggregation,insertion


## Insertions

In [9]:
import gc
import torch
from tqdm import tqdm

variant_scores_dic_insertions = {}

for i, variant in enumerate(tqdm(variant_insertions)):
    try:
        with torch.inference_mode():
            variant_scores_dic_insertions[variant] = score_variant_pytorch(variant)

    except torch.cuda.OutOfMemoryError:
        print(f"CUDA OOM at {variant}")
        torch.cuda.empty_cache()
        gc.collect()
        variant_scores_dic_insertions[variant] = None
        continue

    # clear cache every few variants
    if i % 5 == 0:
        torch.cuda.empty_cache()
        gc.collect()

100%|██████████| 100/100 [15:41<00:00,  9.42s/it]


In [10]:
insertions_results_pytorch = []

for variant in variant_scores_dic_insertions:
    df = variant_scores_dic_insertions[variant]

    if df is None or df.empty:
        continue

    insertions_results_pytorch.append(df)

# combine all into one DataFrame
insertions_results_pytorch = pd.concat(insertions_results_pytorch, ignore_index=True)

In [11]:
# filter out those with no gtx data

insertions_results_pytorch = insertions_results_pytorch[insertions_results_pytorch["gtex_tissue"] != ""]

In [12]:

cols = ["variant_id", "scored_interval", "ontology_curie", "gene_id", "gtex_tissue", "raw_score"]

df_save_insertions = insertions_results_pytorch[cols].copy()

df_save_insertions["variant_id"] = df_save_insertions["variant_id"].astype(str)
df_save_insertions["scored_interval"] = df_save_insertions["scored_interval"].astype(str)

df_save_insertions.to_feather("results/indel_compare/pytorch_insertions.feather")


In [14]:
df_save_insertions = pd.read_feather("results/indel_compare/pytorch_insertions.feather")

In [15]:
df_save_insertions

,variant_id,scored_interval,ontology_curie,gene_id,gtex_tissue,raw_score
567,chr2:60461136:G>GCCA,chr2:60395600-60526672,EFO:0000572,ENSG00000233953,Cells_EBV-transformed_lymphocytes,-0.140644
570,chr2:60461136:G>GCCA,chr2:60395600-60526672,EFO:0002009,ENSG00000233953,Cells_Cultured_fibroblasts,-0.007325
588,chr2:60461136:G>GCCA,chr2:60395600-60526672,UBERON:0000007,ENSG00000233953,Pituitary,-0.030317
590,chr2:60461136:G>GCCA,chr2:60395600-60526672,UBERON:0000458,ENSG00000233953,Cervix_Endocervix,-0.019874
591,chr2:60461136:G>GCCA,chr2:60395600-60526672,UBERON:0000473,ENSG00000233953,Testis,-0.049222
...,...,...,...,...,...,...
513019,chr1:68446872:G>GGGA,chr1:68381336-68512408,None,ENSG00000024526,None,0.000000
513020,chr1:68446872:G>GGGA,chr1:68381336-68512408,None,ENSG00000024526,None,0.000000
513021,chr1:68446872:G>GGGA,chr1:68381336-68512408,None,ENSG00000024526,None,0.000000
513022,chr1:68446872:G>GGGA,chr1:68381336-68512408,None,ENSG00000024526,None,0.000000


## Deletions

In [16]:

import gc
import torch
from tqdm import tqdm

variant_scores_dic_deletions = {}

for i, variant in enumerate(tqdm(variant_deletions)):
    try:
        with torch.inference_mode():
            variant_scores_dic_deletions[variant] = score_variant_pytorch(variant)

    except torch.cuda.OutOfMemoryError:
        print(f"CUDA OOM at {variant}")
        torch.cuda.empty_cache()
        gc.collect()
        variant_scores_dic_deletions[variant] = None
        continue

    # clear cache every few variants
    if i % 5 == 0:
        torch.cuda.empty_cache()
        gc.collect()


100%|██████████| 100/100 [20:23<00:00, 12.24s/it]


In [17]:
# only take one gene for each variant

deletions_results_pytorch = []

for variant in variant_scores_dic_deletions:
    df = variant_scores_dic_deletions[variant]
    if df is None or df.empty:
        continue
    deletions_results_pytorch.append(df)

# convert list of rows → DataFrame
deletions_results_pytorch = pd.concat(deletions_results_pytorch, ignore_index=True)
    


In [18]:
# filter out those with no gtx data

deletions_results_pytorch = deletions_results_pytorch[deletions_results_pytorch["gtex_tissue"] != ""]

In [19]:
cols = ["variant_id", "scored_interval", "ontology_curie", "gene_id", "gtex_tissue", "raw_score"]

df_save_deletions = deletions_results_pytorch[cols].copy()

df_save_deletions["variant_id"] = df_save_deletions["variant_id"].astype(str)
df_save_deletions["scored_interval"] = df_save_deletions["scored_interval"].astype(str)

df_save_deletions.to_feather("results/indel_compare/pytorch_deletions.feather")

In [20]:
df_save_deletions = pd.read_feather("results/indel_compare/pytorch_deletions.feather")

In [21]:
df_save_deletions

,variant_id,scored_interval,ontology_curie,gene_id,gtex_tissue,raw_score
567,chr1:37708311:TTTC>T,chr1:37642775-37773847,EFO:0000572,ENSG00000134690,Cells_EBV-transformed_lymphocytes,0.032651
570,chr1:37708311:TTTC>T,chr1:37642775-37773847,EFO:0002009,ENSG00000134690,Cells_Cultured_fibroblasts,0.026693
588,chr1:37708311:TTTC>T,chr1:37642775-37773847,UBERON:0000007,ENSG00000134690,Pituitary,-0.003092
590,chr1:37708311:TTTC>T,chr1:37642775-37773847,UBERON:0000458,ENSG00000134690,Cervix_Endocervix,0.004515
591,chr1:37708311:TTTC>T,chr1:37642775-37773847,UBERON:0000473,ENSG00000134690,Testis,0.014867
...,...,...,...,...,...,...
422395,chr11:45910267:TAGA>T,chr11:45844731-45975803,None,ENSG00000135365,None,0.000000
422396,chr11:45910267:TAGA>T,chr11:45844731-45975803,None,ENSG00000135365,None,0.000000
422397,chr11:45910267:TAGA>T,chr11:45844731-45975803,None,ENSG00000135365,None,0.000000
422398,chr11:45910267:TAGA>T,chr11:45844731-45975803,None,ENSG00000135365,None,0.000000
